# DichVideo Batch VoxCPM2 from SRT

Upload many `.srt` files and one reference voice audio. For each SRT, this creates one full audio file and one audio file per segment using VoxCPM2.

Use only with your own voice or with clear permission from the voice owner. Runtime > Change runtime type > GPU before running.


In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install -U voxcpm soundfile faster-whisper


In [ ]:
from google.colab import files
from pathlib import Path
import shutil

SRT_DIR = Path('/content/dichvideo_srt_uploads')
OUTPUT_DIR = Path('/content/dichvideo_voxcpm2_audio')
shutil.rmtree(SRT_DIR, ignore_errors=True)
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
SRT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
REF_AUDIO = None
for name, data in uploaded.items():
    suffix = Path(name).suffix.lower()
    if suffix == '.srt':
        (SRT_DIR / name).write_bytes(data)
    elif suffix in {'.mp3', '.wav', '.m4a', '.flac', '.ogg'}:
        ref_path = Path('/content') / name
        ref_path.write_bytes(data)
        REF_AUDIO = str(ref_path)

if not REF_AUDIO:
    raise RuntimeError('Upload one reference audio file, e.g. audio-truyen.mp3')

print('Reference audio:', REF_AUDIO)
print('SRT files:')
for path in sorted(SRT_DIR.glob('*.srt')):
    print('-', path.name)


In [ ]:
%%writefile /content/colab_batch_voxcpm2_from_srt.py
from __future__ import annotations

import argparse
import gc
import json
import logging
import re
import shutil
import subprocess
from pathlib import Path


def main() -> None:
    parser = argparse.ArgumentParser(description="Batch VoxCPM2 TTS from uploaded SRT files.")
    parser.add_argument("--srt-dir", required=True)
    parser.add_argument("--output-dir", required=True)
    parser.add_argument("--ref-audio", required=True)
    parser.add_argument("--model", default="openbmb/VoxCPM2")
    parser.add_argument("--timing-mode", default="no_cut_sequential", choices=["fit_segments", "no_cut_sequential"])
    parser.add_argument("--max-tempo", type=float, default=1.35)
    parser.add_argument("--reference-start", type=float, default=0.0)
    parser.add_argument("--reference-duration", type=float, default=12.0)
    parser.add_argument("--asr-model", default="small")
    parser.add_argument("--language", default="vi")
    parser.add_argument("--cfg-value", type=float, default=2.0)
    parser.add_argument("--inference-timesteps", type=int, default=10)
    parser.add_argument("--use-ultimate-cloning", action="store_true")
    parser.add_argument("--control-style", default="")
    args = parser.parse_args()

    srt_dir = Path(args.srt_dir)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    logger = _setup_logger(output_dir / "batch_voxcpm2_from_srt.log")
    _check_binary("ffmpeg")
    _check_binary("ffprobe")

    srt_files = sorted(srt_dir.glob("*.srt"))
    if not srt_files:
        raise RuntimeError(f"No .srt files found in {srt_dir}")

    ref_wav = _prepare_reference_audio(
        Path(args.ref_audio),
        output_dir / "reference_24k.wav",
        args.reference_start,
        args.reference_duration,
        logger,
    )
    ref_text = _transcribe_reference(ref_wav, args.asr_model, args.language, logger)

    import torch
    from voxcpm import VoxCPM
    import soundfile as sf

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    logger.info("Loading VoxCPM2 model=%s", args.model)
    model = VoxCPM.from_pretrained(args.model, load_denoiser=False)
    sample_rate = int(getattr(getattr(model, "tts_model", None), "sample_rate", 24000))

    for srt_index, srt_path in enumerate(srt_files, start=1):
        logger.info("Processing SRT %s/%s: %s", srt_index, len(srt_files), srt_path)
        segments = parse_srt(srt_path.read_text(encoding="utf-8-sig"))
        if not segments:
            logger.warning("Skipping empty SRT: %s", srt_path)
            continue
        _process_one_srt(
            model=model,
            sf=sf,
            sample_rate=sample_rate,
            srt_path=srt_path,
            segments=segments,
            ref_wav=ref_wav,
            ref_text=ref_text,
            output_dir=output_dir,
            timing_mode=args.timing_mode,
            max_tempo=args.max_tempo,
            cfg_value=args.cfg_value,
            inference_timesteps=args.inference_timesteps,
            use_ultimate_cloning=args.use_ultimate_cloning,
            control_style=args.control_style,
            logger=logger,
        )

    zip_base = output_dir.parent / "voxcpm2_audio_results"
    if zip_base.with_suffix(".zip").exists():
        zip_base.with_suffix(".zip").unlink()
    shutil.make_archive(str(zip_base), "zip", output_dir)
    logger.info("Created zip: %s.zip", zip_base)


def _process_one_srt(model, sf, sample_rate: int, srt_path: Path, segments: list[dict], ref_wav: Path, ref_text: str, output_dir: Path, timing_mode: str, max_tempo: float, cfg_value: float, inference_timesteps: int, use_ultimate_cloning: bool, control_style: str, logger: logging.Logger) -> None:
    name = _safe_stem(srt_path)
    per_srt_dir = output_dir / name
    raw_dir = per_srt_dir / "segments"
    mix_dir = per_srt_dir / "mix_segments"
    raw_dir.mkdir(parents=True, exist_ok=True)
    mix_dir.mkdir(parents=True, exist_ok=True)

    scheduled = []
    cursor = 0.0
    for seg in segments:
        text = " ".join(seg["text"].split())
        if control_style.strip():
            text = control_style.strip() + " " + text
        raw_wav = raw_dir / f"{seg['index']:04d}.wav"
        trim_wav = raw_dir / f"{seg['index']:04d}_trim.wav"
        mix_wav = mix_dir / f"{seg['index']:04d}.wav"
        logger.info("VoxCPM2 generate srt=%s segment=%s chars=%s", srt_path.name, seg["index"], len(text))

        if use_ultimate_cloning and ref_text.strip():
            wav = model.generate(
                text=text,
                prompt_wav_path=str(ref_wav),
                prompt_text=ref_text,
                reference_wav_path=str(ref_wav),
                cfg_value=cfg_value,
                inference_timesteps=inference_timesteps,
            )
        else:
            wav = model.generate(
                text=text,
                reference_wav_path=str(ref_wav),
                cfg_value=cfg_value,
                inference_timesteps=inference_timesteps,
            )
        sf.write(str(raw_wav), wav, sample_rate)
        _trim_silence(raw_wav, trim_wav, logger)

        original_duration = max(0.1, seg["end"] - seg["start"])
        raw_duration = _duration(trim_wav, logger)
        if timing_mode == "fit_segments":
            _fit_audio(trim_wav, mix_wav, original_duration, max_tempo, trim=True, logger=logger)
            scheduled_start = seg["start"]
            scheduled_duration = original_duration
        else:
            tempo = raw_duration / original_duration if original_duration > 0 else 1.0
            if tempo > 1.0:
                _tempo_audio(trim_wav, mix_wav, min(tempo, max_tempo), logger)
            else:
                _convert_audio(trim_wav, mix_wav, logger)
            scheduled_duration = _duration(mix_wav, logger)
            scheduled_start = max(seg["start"], cursor)
            cursor = scheduled_start + scheduled_duration

        scheduled.append({
            **seg,
            "scheduled_start": scheduled_start,
            "scheduled_end": scheduled_start + scheduled_duration,
            "audio_duration": scheduled_duration,
            "raw_audio": str(raw_wav.relative_to(per_srt_dir)),
            "trimmed_audio": str(trim_wav.relative_to(per_srt_dir)),
            "mix_audio": str(mix_wav.relative_to(per_srt_dir)),
        })

    full_wav = per_srt_dir / f"{name}_full.wav"
    _mix_scheduled_audio(scheduled, full_wav, logger, per_srt_dir)
    (per_srt_dir / f"{name}.srt").write_text(srt_path.read_text(encoding="utf-8-sig"), encoding="utf-8")
    _write_json(per_srt_dir / "timing_schedule.json", scheduled)
    logger.info("Finished %s -> %s", srt_path.name, full_wav)


def parse_srt(content: str) -> list[dict]:
    content = content.replace("\r\n", "\n").replace("\r", "\n").strip()
    if not content:
        return []
    blocks = re.split(r"\n\s*\n", content)
    segments = []
    fallback_index = 1
    for block in blocks:
        lines = [line.strip() for line in block.split("\n") if line.strip()]
        if not lines:
            continue
        timing_line_index = next((i for i, line in enumerate(lines) if "-->" in line), None)
        if timing_line_index is None:
            continue
        maybe_index = lines[0] if timing_line_index > 0 else str(fallback_index)
        try:
            index = int(re.sub(r"\D+", "", maybe_index) or fallback_index)
        except ValueError:
            index = fallback_index
        timing = lines[timing_line_index]
        start_s, end_s = [part.strip().split()[0] for part in timing.split("-->", 1)]
        text = " ".join(lines[timing_line_index + 1:]).strip()
        if text:
            segments.append({"index": index, "start": _parse_srt_timestamp(start_s), "end": _parse_srt_timestamp(end_s), "text": text})
            fallback_index += 1
    return segments


def _parse_srt_timestamp(value: str) -> float:
    match = re.match(r"(\d+):(\d+):(\d+)[,.](\d+)", value)
    if not match:
        raise ValueError(f"Invalid SRT timestamp: {value}")
    h, m, s, ms = match.groups()
    return int(h) * 3600 + int(m) * 60 + int(s) + int(ms.ljust(3, "0")[:3]) / 1000


def _prepare_reference_audio(input_path: Path, output_path: Path, start: float, duration: float, logger: logging.Logger) -> Path:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    _run(["ffmpeg", "-y", "-i", str(input_path), "-ss", f"{start:.3f}", "-t", f"{duration:.3f}", "-vn", "-ar", "24000", "-ac", "1", str(output_path)], logger)
    return output_path


def _transcribe_reference(ref_wav: Path, asr_model: str, language: str, logger: logging.Logger) -> str:
    import torch
    from faster_whisper import WhisperModel

    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"
    logger.info("Transcribing reference with faster-whisper model=%s device=%s compute_type=%s", asr_model, device, compute_type)
    asr = WhisperModel(asr_model, device=device, compute_type=compute_type)
    segments, _info = asr.transcribe(str(ref_wav), language=language, vad_filter=True)
    ref_text = " ".join(seg.text.strip() for seg in segments).strip()
    logger.info("REF_TEXT=%s", ref_text)
    del asr
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if not ref_text:
        raise RuntimeError("Whisper did not recognize transcript from reference audio. Use a clearer single-speaker reference.")
    return ref_text


def _trim_silence(input_path: Path, output_path: Path, logger: logging.Logger) -> None:
    _run(["ffmpeg", "-y", "-i", str(input_path), "-af", "silenceremove=start_periods=1:start_duration=0.15:start_threshold=-45dB:stop_periods=1:stop_duration=0.35:stop_threshold=-45dB", str(output_path)], logger)


def _fit_audio(input_path: Path, output_path: Path, target_duration: float, max_tempo: float, trim: bool, logger: logging.Logger) -> None:
    source_duration = _duration(input_path, logger)
    tempo = source_duration / target_duration if target_duration > 0 else 1.0
    audio_filter = f"atempo={min(tempo, max_tempo):.5f},apad" if tempo > 1.0 else "apad"
    if trim:
        audio_filter += f",atrim=0:{target_duration:.3f}"
    _run(["ffmpeg", "-y", "-i", str(input_path), "-filter:a", audio_filter, "-ac", "1", "-ar", "44100", str(output_path)], logger)


def _tempo_audio(input_path: Path, output_path: Path, tempo: float, logger: logging.Logger) -> None:
    _run(["ffmpeg", "-y", "-i", str(input_path), "-filter:a", f"atempo={tempo:.5f}", "-ac", "1", "-ar", "44100", str(output_path)], logger)


def _convert_audio(input_path: Path, output_path: Path, logger: logging.Logger) -> None:
    _run(["ffmpeg", "-y", "-i", str(input_path), "-ac", "1", "-ar", "44100", str(output_path)], logger)


def _mix_scheduled_audio(scheduled: list[dict], output_path: Path, logger: logging.Logger, root_dir: Path) -> None:
    total_duration = max(float(item["scheduled_end"]) for item in scheduled)
    silence_path = root_dir / "_silence.wav"
    _run(["ffmpeg", "-y", "-f", "lavfi", "-i", "anullsrc=channel_layout=mono:sample_rate=44100", "-t", f"{total_duration:.3f}", str(silence_path)], logger)
    inputs = ["-i", str(silence_path)]
    filters = []
    mix_inputs = ["[0:a]"]
    for input_index, item in enumerate(scheduled, start=1):
        audio_path = root_dir / item["mix_audio"]
        inputs.extend(["-i", str(audio_path)])
        delay_ms = max(0, int(float(item["scheduled_start"]) * 1000))
        label = f"a{input_index}"
        filters.append(f"[{input_index}:a]adelay={delay_ms}:all=1[{label}]")
        mix_inputs.append(f"[{label}]")
    filter_complex = ";".join(filters + [f"{''.join(mix_inputs)}amix=inputs={len(mix_inputs)}:normalize=0[out]"])
    _run(["ffmpeg", "-y", *inputs, "-filter_complex", filter_complex, "-map", "[out]", "-ac", "2", "-ar", "44100", str(output_path)], logger)
    silence_path.unlink(missing_ok=True)


def _duration(path: Path, logger: logging.Logger) -> float:
    completed = _run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", str(path)], logger)
    return float(completed.stdout.strip())


def _safe_stem(path: Path) -> str:
    stem = re.sub(r"[^A-Za-z0-9_.-]+", "_", path.stem).strip("._")
    return stem or "srt"


def _run(cmd: list[str], logger: logging.Logger) -> subprocess.CompletedProcess:
    logger.info("Running command: %s", " ".join(cmd))
    completed = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")
    if completed.stdout.strip():
        logger.info("stdout: %s", completed.stdout.strip()[-3000:])
    if completed.stderr.strip():
        logger.info("stderr: %s", completed.stderr.strip()[-3000:])
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed with code {completed.returncode}: {' '.join(cmd)}")
    return completed


def _check_binary(name: str) -> None:
    if shutil.which(name) is None:
        raise RuntimeError(f"Missing dependency: {name}")


def _write_json(path: Path, data) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


def _setup_logger(log_path: Path) -> logging.Logger:
    logger = logging.getLogger("dichvideo_batch_voxcpm2_from_srt")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    formatter = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    file_handler = logging.FileHandler(log_path, encoding="utf-8")
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)
    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)
    return logger


if __name__ == "__main__":
    main()


In [ ]:
TIMING_MODE = 'no_cut_sequential'  # no_cut_sequential or fit_segments
MAX_TEMPO = '1.35'
REFERENCE_START = '0'
REFERENCE_DURATION = '12'
ASR_MODEL = 'small'
LANGUAGE = 'vi'
CFG_VALUE = '2.0'
INFERENCE_TIMESTEPS = '10'
USE_ULTIMATE_CLONING = False  # False gi?p audio ch? ??c text trong SRT, ?t b? l?p/th?m ch? t? prompt.
CONTROL_STYLE = ''  # ?? tr?ng ?? model kh?ng ??c nh?m style instruction th?nh l?i n?i.

ultimate_flag = '--use-ultimate-cloning' if USE_ULTIMATE_CLONING else ''

!python /content/colab_batch_voxcpm2_from_srt.py \
  --srt-dir "$SRT_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --ref-audio "$REF_AUDIO" \
  --timing-mode "$TIMING_MODE" \
  --max-tempo "$MAX_TEMPO" \
  --reference-start "$REFERENCE_START" \
  --reference-duration "$REFERENCE_DURATION" \
  --asr-model "$ASR_MODEL" \
  --language "$LANGUAGE" \
  --cfg-value "$CFG_VALUE" \
  --inference-timesteps "$INFERENCE_TIMESTEPS" \
  --control-style "$CONTROL_STYLE" \
  $ultimate_flag


In [ ]:
from google.colab import files
files.download('/content/voxcpm2_audio_results.zip')
